# Sesión 5 · Aprender por ensayo y error
**Machine Learning y Seguridad en la Nube — Especialización en Big Data (UCompensar)**

Este notebook acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/diapositivas/sesion-5-aprender-por-ensayo-y-error.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/.

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el notebook no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
   Mientras tengan `...` sin reemplazar, las celdas de ejercicio dan error o un mensaje con ✗ (si el error
   menciona `Ellipsis` o `ellipsis`, te quedó un `...`). Con **Entorno de ejecución → Ejecutar todas**, Colab se
   detiene en el primer error (el ejercicio 2): complétalo, o sigue con **Mayús + Enter** celda por celda.
4. La primera celda **descarga** los datos del caso *Yarumo Fibra S.A.S.*, una empresa **ficticia** (sus
   datos se generaron para el curso y no corresponden a personas ni a empresas reales), y comprueba que
   el archivo sea idéntico al del curso. Si la descarga falla, la celda te dice cómo subirlo a mano.
5. Hoy todo es **numpy**, **pandas** y **matplotlib**: scikit-learn no trae aprendizaje por refuerzo. El notebook entero
   corre en menos de un minuto, sin GPU. Cada cálculo al azar lleva su semilla, así que tus cifras deben
   salir iguales a las de las diapositivas.

En el código, los decimales van con punto (así los escribe Python); en el texto, con coma.

In [ ]:
#@title ▶ Descarga los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Yarumo Fibra S.A.S.» (empresa FICTICIA), descargados del repositorio del curso.

Deja en datos/clientes.csv los 8000 clientes del curso y comprueba su huella SHA-256: si el
archivo no es idéntico al del curso, no se usa. Solo usa la biblioteca estándar de Python.
"""

import hashlib
import os
import urllib.request
from pathlib import Path

DATOS_URL = "https://raw.githubusercontent.com/JotaMao1985/Machine-Learning-cloud-computing_Ucompensar/main/datos/clientes.csv"
DATOS_SHA256 = "2d4fad61a49efcf1d071b6c412e5224e71eca23047cbbac6b4017c8627f9c93b"
PORTADA = "https://jotamao1985.github.io/Machine-Learning-cloud-computing_Ucompensar/"


def _es_del_curso(contenido):
    return hashlib.sha256(contenido).hexdigest() == DATOS_SHA256


def _sin_datos(motivo):
    return (f"No pude dejar listos los datos: {motivo}.\n"
            f"Descarga clientes.csv desde la portada del curso ({PORTADA}), súbelo a Colab con el "
            "ícono de carpeta de la izquierda (Subir al almacenamiento de la sesión) y vuelve a "
            "ejecutar esta celda.")


def _leer_url():
    # Solo al preparar el material: la variable apunta a una copia local del mismo archivo
    local = os.environ.get("YARUMO_CSV_LOCAL")
    if local:
        return Path(local).read_bytes()
    with urllib.request.urlopen(DATOS_URL, timeout=30) as respuesta:
        return respuesta.read()


def descargar_datos(carpeta="datos"):
    """Deja <carpeta>/clientes.csv, idéntico al del curso, y devuelve su ruta."""
    ruta = Path(carpeta) / "clientes.csv"
    subido = Path("clientes.csv")  # por si se subió a mano a la carpeta de trabajo de Colab
    if ruta.exists() and _es_del_curso(ruta.read_bytes()):
        contenido, origen = ruta.read_bytes(), "ya estaban"
    elif subido.exists() and _es_del_curso(subido.read_bytes()):
        contenido, origen = subido.read_bytes(), "copiados del archivo que subiste"
    else:
        try:
            contenido = _leer_url()
        except OSError as error:
            raise RuntimeError(_sin_datos(f"no se pudo descargar el archivo ({error})")) from None
        if not _es_del_curso(contenido):
            raise RuntimeError(_sin_datos("el archivo descargado no es el del curso"))
        origen = "descargados del repositorio del curso"
    if origen != "ya estaban":
        ruta.parent.mkdir(parents=True, exist_ok=True)
        temporal = ruta.with_suffix(".tmp")  # sin archivos a medias si algo falla al escribir
        temporal.write_bytes(contenido)
        temporal.replace(ruta)
    lineas = contenido.decode("utf-8").splitlines()
    print(f"Listo: {ruta} con {len(lineas) - 1} clientes y {len(lineas[0].split(','))} columnas, {origen}")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras descarga_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2 y scikit-learn 1.6: si Colab trae otras, se dice."""
    import pandas as pd
    import sklearn

    if pd.__version__.split(".")[:2] != ["2", "2"] or sklearn.__version__.split(".")[:2] != ["1", "6"]:
        print(f"Nota: este Colab trae pandas {pd.__version__} y scikit-learn {sklearn.__version__}; "
              "las diapositivas se hicieron con pandas 2.2 y scikit-learn 1.6. Si alguna salida "
              "se ve un poco distinta, es por eso.")


_aviso_version()


"""Sesión 5: estilo de las gráficas del curso y los verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con descarga_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante hizo, dice qué está bien y, en lo demás, da una pista sin
la respuesta. Nunca se cae: un «...» sin reemplazar, un error dentro de la función del estudiante o
un tipo inesperado se avisa.

Los verificadores NO usan las funciones del notebook (el estudiante puede haberlas cambiado):
tienen aquí su propia copia de la rejilla y del simulador de Yarumo, con los mismos datos:

- ejercicio 1: prueban la función `actualiza` del estudiante con casos fijos en los que cada error
  típico da un valor distinto (sin la recompensa, sin descontar, sin lo mejor del estado siguiente,
  sin alfa, sin restar el valor actual, solo el cambio, alfa y gamma escritos a mano, sin tratar el
  final);
- ejercicio 2: entrenan aquí los dos agentes de la rejilla (ε = 0 y ε = 0,1, semilla 2026, 500
  episodios), comparan lo que el estudiante dice que pasó y revisan la letra de la explicación;
- ejercicio 3: llaman a la política del estudiante para los 24 meses y recalculan su valor exacto.
"""

_FILAS, _COLUMNAS = 4, 4
_INICIO, _META, _POZOS = (3, 0), (0, 3), ((1, 1), (2, 3))
_MOV = ((-1, 0), (0, 1), (1, 0), (0, -1))   # arriba, derecha, abajo, izquierda
_RAZON_BIEN = "C"


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F", "#0E7490", "#CA8A04"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 13,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()

_CACHE = {}


def _seguro(verificador):
    """Red de seguridad: si algo imprevisto falla al revisar, se avisa sin traceback."""
    import functools

    @functools.wraps(verificador)
    def envoltura(*args, **kwargs):
        try:
            return verificador(*args, **kwargs)
        except Exception as error:   # noqa: BLE001 — el estudiante nunca ve un traceback
            print(f"✗  No pude revisar lo que entregaste ({type(error).__name__}). Revisa lo que pasaste al "
                  "verificador y vuelve a ejecutar la celda")
    return envoltura


def _coma(x, decimales=2):
    """Un número con coma decimal, para los mensajes."""
    return f"{x:.{decimales}f}".replace(".", ",").replace("-", "−")


def _sobrantes(sueltos, otros, nombres, ejemplo):
    """Avisa de valores sin nombre o con otro nombre. True si hubo alguno."""
    if sueltos:
        print(f"✗  Pasa los valores por nombre, como en la plantilla: {ejemplo}")
        return True
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    return False


def _un_numero(x):
    """(valor, None) si x es un solo número real; si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np

    if x is Ellipsis:
        return None, "todavía tiene los puntos suspensivos (...) de la plantilla"
    if x is None:
        return None, "no devuelve nada (None): ¿falta el return?"
    if isinstance(x, np.ndarray) and x.size == 1:
        x = x.ravel()[0]
    if isinstance(x, (np.ndarray, list, tuple)):
        return None, (f"devuelve {len(x)} valores; hace falta uno solo, el valor nuevo de Q en esa casilla y acción. "
                      "El valor de la casilla s y la acción a se escribe Q[s][a]")
    if isinstance(x, (bool, np.bool_)):
        return None, "devuelve True o False; hace falta un número"
    if isinstance(x, str):
        return None, "devuelve texto; hace falta un número"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no devuelve un número real (devuelve {type(x).__name__})"
    valor = float(x)
    if math.isnan(valor) or math.isinf(valor):
        return None, "devuelve nan o infinito: revisa la cuenta"
    return valor, None


# ---------------------------------------------------------------------------------------------
# La rejilla (copia propia de los verificadores)
# ---------------------------------------------------------------------------------------------

def _paso(s, a):
    f = min(max(s[0] + _MOV[a][0], 0), _FILAS - 1)
    c = min(max(s[1] + _MOV[a][1], 0), _COLUMNAS - 1)
    s2 = (f, c)
    if s2 == _META:
        return s2, 10.0, True
    if s2 in _POZOS:
        return s2, -10.0, True
    return s2, 0.0, False


def _entrena(eps, episodios=500, alfa=0.1, gamma=0.9, semilla=2026):
    """El mismo entrenamiento de la celda [entrenar]."""
    import numpy as np

    rng = np.random.default_rng(semilla)
    Q = np.zeros((_FILAS, _COLUMNAS, 4))
    visitas = np.zeros((_FILAS, _COLUMNAS), dtype=int)
    for _ in range(episodios):
        s = _INICIO
        for _t in range(50):
            visitas[s] += 1
            if rng.random() < eps:
                a = int(rng.integers(4))
            else:
                a = int(np.argmax(Q[s]))
            s2, r, fin = _paso(s, a)
            mejor_despues = 0 if fin else Q[s2].max()
            Q[s][a] += alfa * (r + gamma * mejor_despues - Q[s][a])
            s = s2
            if fin:
                break
    return Q, visitas


def _llega(Q):
    import numpy as np

    s = _INICIO
    for _ in range(50):
        s, _r, fin = _paso(s, int(np.argmax(Q[s])))
        if fin:
            return s == _META
    return False


# ---------------------------------------------------------------------------------------------
# Ejercicio 1 · La actualización de Q
# ---------------------------------------------------------------------------------------------

def _probar(funcion, Q, s, a, r, s2, fin, alfa, gamma):
    """Llama a la función del estudiante con una copia de Q. Devuelve (valor, motivo, cambió_Q)."""
    import numpy as np

    import inspect

    copia = Q.copy()
    try:
        inspect.signature(funcion).bind(copia, s, a, r, s2, fin, alfa=alfa, gamma=gamma)
    except TypeError:
        return None, ("no recibe los mismos argumentos de la plantilla: deja la primera línea como "
                      "estaba, def actualiza(Q, s, a, r, s2, fin, alfa=0.1, gamma=0.9):"), False
    except ValueError:
        pass   # una función sin firma legible: se prueba igual
    try:
        resultado = funcion(copia, s, a, r, s2, fin, alfa=alfa, gamma=gamma)
    except TypeError as error:
        texto = str(error)
        if "llipsis" in texto:
            return None, "todavía tiene los puntos suspensivos (...) de la plantilla: reemplázalos", False
        return None, f"dio un error al ejecutarla (TypeError: {texto})", False
    except SystemExit:
        return None, "detuvo la ejecución (SystemExit): quita el exit() o el quit()", False
    except NameError as error:
        return None, (f"usa un nombre que no existe dentro de la función ({error}): usa solo Q, s, a, r, s2, "
                      "fin, alfa, gamma y lo_mejor_despues"), False
    except Exception as error:   # noqa: BLE001
        return None, f"dio un error al ejecutarla ({type(error).__name__}: {error})", False
    cambio = not np.array_equal(copia, Q)
    valor, motivo = _un_numero(resultado)
    return valor, motivo, cambio


@_seguro
def revisar_actualiza(actualiza=...):
    """Ejercicio 1: la función que devuelve el valor nuevo de Q[s][a]."""
    import numpy as np

    if actualiza is Ellipsis or not callable(actualiza):
        print("✗  Pasa la función, sin paréntesis: revisar_actualiza(actualiza)")
        return
    # Caso 1: Q[s][a] = 2,0; r = −1; lo mejor del estado siguiente = 5,0; α = 0,1; γ = 0,9
    Q = np.zeros((_FILAS, _COLUMNAS, 4))
    s, a, s2 = (2, 0), 2, (1, 0)
    Q[s] = [3.0, 0.0, 2.0, -1.0]      # Q[s][a] = 2,0, que no es el máximo de su casilla
    Q[s2] = [1.0, 0.0, -3.0, 5.0]     # el máximo de la casilla siguiente no está en la acción a
    v, motivo, cambio = _probar(actualiza, Q, s, a, -1.0, s2, False, 0.1, 0.9)
    if motivo:
        print(f"✗  Tu función {motivo}")
        return
    if cambio:
        print("·  Tu función cambia la tabla Q que recibe. No hace falta: basta con que devuelva el valor nuevo")
    errores = {
        2.15: None,
        2.25: "el objetivo no incluye la recompensa que el agente acaba de recibir (r)",
        2.20: "lo que espera ganar después entra completo al objetivo: falta descontarlo con gamma",
        1.70: ("al objetivo le falta lo que espera ganar después: lo mejor del estado siguiente "
               "(lo_mejor_despues)"),
        3.50: "el valor salta de una vez hasta el objetivo: revisa el papel de alfa en la regla",
        2.35: ("revisa qué multiplica alfa: no es el objetivo entero, sino cuánto le falta al valor actual para "
               "llegar a él (con sus signos)"),
        0.15: "devuelve solo el cambio; falta sumarlo al valor actual",
        1.85: "resta en el orden contrario: lo que le falta al valor actual es el objetivo menos el valor actual",
    }
    pista = next((p for valor, p in errores.items() if abs(v - valor) < 1e-6), "sin pista")
    if pista is None:
        print("✓  Con un caso a mitad del camino: bien")
    else:
        print(f"✗  Con un caso a mitad del camino, tu función da {_coma(v)}. "
              + (pista[0].upper() + pista[1:] if pista != "sin pista" else
                 "Repasa la regla: el valor actual más alfa por (el objetivo menos el valor actual), y el "
                 "objetivo es la recompensa más gamma por lo mejor del estado siguiente"))
        return
    # Caso 2: los mismos datos con α = 0,5 y γ = 0,5 (¿usa los parámetros o números escritos a mano?)
    v2, motivo, _ = _probar(actualiza, Q, s, a, -1.0, s2, False, 0.5, 0.5)
    if motivo or abs(v2 - 1.75) > 1e-6:
        print("✗  Con otra alfa y otra gamma, tu función da lo mismo o no funciona: usa los parámetros alfa y "
              "gamma, no 0.1 ni 0.9 escritos a mano")
        return
    print("✓  Con otra tasa de aprendizaje y otro factor de descuento: bien")
    # Caso 3: el paso que termina (fin=True), con una fila no nula en el estado siguiente
    Qf = np.zeros((_FILAS, _COLUMNAS, 4))
    sf, af = (0, 2), 2
    Qf[sf] = [3.0, 0.0, 1.0, 0.0]
    Qf[_META] = [4.0, 4.0, 4.0, 4.0]
    v3, motivo, _ = _probar(actualiza, Qf, sf, af, 10.0, _META, True, 0.1, 0.9)
    if motivo:
        print(f"✗  En el último paso de un episodio, tu función {motivo}")
        return
    if abs(v3 - 1.9) > 1e-6:
        print("✗  En el último paso de un episodio (fin es True) no da lo esperado: al llegar a la meta o a un "
              "pozo ya no hay después, y lo que espera ganar después vale 0")
        return
    print("✓  En el último paso de un episodio: bien")
    print("\n¡Listo! Tu función hace lo mismo que la línea de la actualización en la celda [entrenar]: el valor "
          "se mueve una fracción alfa hacia el objetivo, r + gamma · (lo mejor de después).")


# ---------------------------------------------------------------------------------------------
# Ejercicio 2 · Sin explorar
# ---------------------------------------------------------------------------------------------

def _un_logico(x, nombre):
    import numpy as np

    if x is Ellipsis:
        return None, f"{nombre}: todavía tiene los puntos suspensivos (...) de la plantilla"
    if isinstance(x, (bool, np.bool_)):
        return bool(x), None
    if isinstance(x, str):
        return None, f"{nombre}: escribe True o False sin comillas (es un valor lógico, no un texto)"
    if isinstance(x, (int, np.integer)) and x in (0, 1):
        return None, f"{nombre}: escribe True o False, no 0 ni 1"
    return None, f"{nombre}: hace falta True o False, lo que devuelve llega_a_la_meta(Q)"


@_seguro
def revisar_exploracion(*sueltos, llega_eps0=..., llega_eps01=..., casillas_eps0=..., razon=..., **otros):
    """Ejercicio 2: ¿llega cada agente a la meta? ¿Por qué el que no explora no llega?"""
    nombres = ["llega_eps0", "llega_eps01", "casillas_eps0", "razon"]
    if _sobrantes(sueltos, otros, nombres,
                  "revisar_exploracion(llega_eps0=llega_eps0, llega_eps01=llega_eps01, "
                  "casillas_eps0=casillas_eps0, razon=razon)"):
        return
    if "rejilla" not in _CACHE:
        q0, v0 = _entrena(0.0)
        q01, _v01 = _entrena(0.1)
        _CACHE["rejilla"] = {"0": _llega(q0), "01": _llega(q01), "visitadas0": int((v0 > 0).sum()),
                             "pasos0": int(v0.sum())}
    ref = _CACHE["rejilla"]
    bien = 0
    for nombre, x, clave in (("llega_eps0", llega_eps0, "0"), ("llega_eps01", llega_eps01, "01")):
        valor, motivo = _un_logico(x, nombre)
        if motivo:
            print(f"✗  {motivo}")
        elif valor == ref[clave]:
            print(f"✓  {nombre}: bien")
            bien += 1
        else:
            print(f"✗  {nombre}: no es lo que pasa con ese agente. ¿Lo entrenaste con el ε que dice su nombre y la "
                  "semilla de la plantilla? ¿Le pasaste a llega_a_la_meta la tabla Q de ese mismo agente?")
    import numbers
    import numpy as np

    if casillas_eps0 is Ellipsis:
        print("✗  casillas_eps0: todavía tiene los puntos suspensivos (...)")
    elif isinstance(casillas_eps0, np.ndarray) and casillas_eps0.size > 1:
        print("✗  casillas_eps0: es una tabla; hace falta un solo número: súmala con .sum()")
    elif isinstance(casillas_eps0, (bool, np.bool_)) or not isinstance(casillas_eps0, (numbers.Integral, np.integer)):
        print("✗  casillas_eps0: hace falta un número entero, cuántas casillas tienen al menos una visita")
    elif int(casillas_eps0) == ref["visitadas0"]:
        print("✓  casillas_eps0: bien")
        bien += 1
    elif int(casillas_eps0) == ref["pasos0"]:
        print("✗  casillas_eps0: ese es el total de pasos; se piden las casillas distintas con al menos una visita")
    else:
        print("✗  casillas_eps0: no es lo que da el agente sin exploración: cuenta las casillas de visitas_0 que "
              "no están en 0")
    pistas = {
        "A": ("La tasa de aprendizaje es la misma en los dos agentes: lo único que cambia es ε. Mira las visitas "
              "del agente que no explora: ¿por cuáles casillas pasó?"),
        "B": ("Con γ = 0,9, llegar a la meta en 6 pasos todavía vale 10 · 0,9⁵ ≈ 5,9, más que cero: sí vale la "
              "pena ir. Mira las visitas del agente que no explora: ¿llegó alguna vez cerca de la meta?"),
        "D": ("Pruébalo: entrena el agente con eps=0.0 y episodios=5000. ¿Llega ahora? Mira también cuántas "
              "casillas distintas visitó"),
    }
    if razon is Ellipsis:
        print("✗  razon: todavía tiene los puntos suspensivos (...): escribe la letra entre comillas, por ejemplo \"A\"")
    elif not isinstance(razon, str) or razon.strip().upper().strip(".)") not in ("A", "B", "C", "D"):
        print("✗  razon: escribe solo la letra de la explicación, entre comillas: \"A\", \"B\", \"C\" o \"D\"")
    else:
        letra = razon.strip().upper().strip(".)")
        if letra == _RAZON_BIEN:
            print("✓  razon: bien")
            bien += 1
        else:
            print(f"✗  razon: no es esa. {pistas[letra]}")
    if bien == 4:
        print(f"\n¡Listo! Sin explorar, el agente visitó solo {ref['visitadas0']} casillas en 500 episodios: como "
              "nunca recibió una recompensa, su tabla quedó en ceros y eligió siempre la misma acción. Con "
              "ε = 0,1, las jugadas al azar lo llevan alguna vez a la meta, y desde ahí la recompensa viaja hacia atrás.")


# ---------------------------------------------------------------------------------------------
# Ejercicio 3 · Tu política de retención (simulador propio, con los mismos datos y supuestos)
# ---------------------------------------------------------------------------------------------

def _yarumo():
    """El riesgo mensual por tramo, la factura y los supuestos, como en las celdas [riesgo] y [entorno]."""
    if "yarumo" in _CACHE:
        return _CACHE["yarumo"]
    import pandas as pd

    try:
        clientes = pd.read_csv("datos/clientes.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/clientes.csv: ejecuta primero la celda que descarga los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None
    ant = clientes["antiguedad_meses"]
    sel = clientes[((ant <= 12) & (clientes["descuento"] == "Sí"))
                   | (ant.between(13, 24) & (clientes["descuento"] == "No"))]
    tramo = pd.cut(sel["antiguedad_meses"], [0, 9, 12, 24], labels=False)
    a3 = (sel["cancelo"] == "Sí").groupby(tramo).mean().to_numpy()
    _CACHE["yarumo"] = {"riesgo": 1 - (1 - a3) ** (1 / 3),
                        "factura": round(clientes["factura_mensual"].median() / 1000),
                        "baja": (0.0, 0.25, 0.60), "costo": (0.0, 20.0, 60.0)}
    return _CACHE["yarumo"]


def _valor_exacto(acciones):
    """Valor esperado por cliente (miles de pesos) de una lista de 24 acciones, de atrás hacia adelante."""
    y = _yarumo()
    valor = 0.0
    for mes in range(24, 0, -1):
        a = acciones[mes - 1]
        k = 0 if mes <= 9 else (1 if mes <= 12 else 2)
        riesgo = y["riesgo"][k] * (1 - y["baja"][a])
        valor = -y["costo"][a] + (1 - riesgo) * (y["factura"] + valor)
    return valor


def _acciones_de(politica):
    """Las 24 acciones de la política (función del mes o lista), o (None, motivo)."""
    import numbers
    import numpy as np

    if politica is Ellipsis:
        return None, "tu política todavía tiene los puntos suspensivos (...)"
    if callable(politica):
        acciones = []
        for mes in range(1, 25):
            try:
                a = politica(mes)
            except TypeError as error:
                if "llipsis" in str(error):
                    return None, "tu política todavía tiene los puntos suspensivos (...): reemplázalos"
                return None, f"tu política dio un error en el mes {mes} (TypeError: {error})"
            except Exception as error:   # noqa: BLE001
                return None, f"tu política dio un error en el mes {mes} ({type(error).__name__}: {error})"
            acciones.append(a)
    elif isinstance(politica, (list, tuple, np.ndarray)):
        acciones = list(np.ravel(politica)) if isinstance(politica, np.ndarray) else list(politica)
        if len(acciones) != 24:
            return None, f"tu lista tiene {len(acciones)} acciones; hacen falta 24, una por mes"
    else:
        return None, "pasa tu política, la función, sin paréntesis: revisar_politica(mi_politica, valor=valor)"
    for mes, a in enumerate(acciones, start=1):
        if a is Ellipsis:
            return None, f"en el mes {mes} tu política devuelve los puntos suspensivos (...) de la plantilla"
        if a is None:
            return None, f"en el mes {mes} tu política no devuelve nada: ¿falta un return?"
        if isinstance(a, str):
            return None, (f"en el mes {mes} tu política devuelve el texto {a!r}: devuelve el número de la acción, "
                          "0 (nada), 1 (llamar) o 2 (oferta de retención)")
        if isinstance(a, (bool, np.bool_)) or not isinstance(a, (numbers.Integral, np.integer)):
            return None, (f"en el mes {mes} tu política devuelve {a!r}: devuelve 0, 1 o 2, números enteros")
        if int(a) not in (0, 1, 2):
            return None, f"en el mes {mes} tu política devuelve {a}: las acciones son 0, 1 y 2"
    return [int(a) for a in acciones], None


@_seguro
def revisar_politica(politica=..., *sueltos, valor=..., **otros):
    """Ejercicio 3: una política de retención válida y su valor, calculado con valor_politica."""
    import numpy as np

    if _sobrantes(sueltos, otros, ["valor"], "revisar_politica(mi_politica, valor=valor)"):
        return
    if _yarumo() is None:
        return
    acciones, motivo = _acciones_de(politica)
    if motivo:
        print(f"✗  {motivo}")
        return
    print("✓  Tu política da una acción válida en los 24 meses")
    nombres = {0: "nada", 1: "llamar", 2: "oferta de retención"}
    tramos = []
    inicio = 1
    for mes in range(2, 26):
        if mes == 25 or acciones[mes - 1] != acciones[inicio - 1]:
            fin = mes - 1
            tramos.append(f"{nombres[acciones[inicio - 1]]} en {'el mes ' + str(inicio) if inicio == fin else f'los meses {inicio}–{fin}'}")
            inicio = mes
    print("   Tu plan: " + "; ".join(tramos))
    exacto = _valor_exacto(acciones)
    v, problema = _un_numero(valor)
    if problema:
        if valor is Ellipsis:
            problema = "todavía tiene los puntos suspensivos (...)"
        elif valor is None:
            problema = "está vacío (None)"
        elif isinstance(valor, (list, tuple, np.ndarray)):
            problema = "es una lista o una tabla; hace falta un solo número"
        elif isinstance(valor, str):
            problema = "es texto; hace falta un número"
        else:
            problema = "no es un número"
        print(f"✗  valor: {problema}. Calcúlalo con valor_politica(mi_politica)")
        return
    if abs(v - exacto) <= 0.05:
        print("✓  valor: bien")
    elif abs(v - round(exacto)) < 1e-9 or abs(v - round(exacto, 1)) < 1e-9:
        print("✓  valor: bien (redondeado)")
    else:
        otros_valores = {"no hacer nada": [0] * 24, "llamar siempre": [1] * 24, "la oferta siempre": [2] * 24,
                         "la oferta en los meses 10–12": [2 if 10 <= m <= 12 else 0 for m in range(1, 25)]}
        de_otra = next((n for n, acc in otros_valores.items() if acc != acciones and abs(_valor_exacto(acc) - v) < 0.06),
                       None)
        if de_otra:
            print(f"✗  valor: es el de otra política ({de_otra}); calcula el de la tuya con valor_politica(mi_politica)")
        else:
            print("✗  valor: no coincide con el de tu política. No lo escribas a mano: calcúlalo con "
                  "valor_politica(mi_politica)")
        return
    regla = _valor_exacto([2 if 10 <= m <= 12 else 0 for m in range(1, 25)])
    print(f"\n¡Listo! Bajo los supuestos del simulador, tu política deja en promedio {_coma(exacto, 1)} mil pesos por "
          f"cliente en 24 meses; la oferta solo en los meses 10–12 deja {_coma(regla, 1)} mil pesos. Compárala con la mejor "
          "posible, en la celda [mejor], y explica en «Tus respuestas» por qué la tuya gana o pierde.")


# ---------------------------------------------------------------------------------------------
# Dibujos (los llama el notebook; no hace falta leerlos)
# ---------------------------------------------------------------------------------------------

def dibujar_rejilla(Q=None, visitas=None, titulo=""):
    """La rejilla: la salida, la meta, los pozos y, si se pasa Q, la mejor acción de cada casilla
    (una flecha) con las veces que el agente pasó por ella debajo. Un punto gris: empate, nada aprendido."""
    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib.patches import Rectangle

    fig, eje = plt.subplots(figsize=(4.6, 4.6) if Q is None else (5.2, 5.2))
    for f in range(_FILAS):
        for c in range(_COLUMNAS):
            s = (f, c)
            if s == _META:
                fondo, texto, color = "#FDE5D3", "Meta\n+10", "#B44700"
            elif s in _POZOS:
                fondo, texto, color = "#566B7F", "Pozo\n−10", "#FFFFFF"
            elif s == _INICIO:
                fondo, texto, color = "#EFE3F3", "Salida", "#6B2179"
            else:
                fondo, texto, color = "#FFFFFF", "", "#1E293B"
            eje.add_patch(Rectangle((c, f), 1, 1, facecolor=fondo, edgecolor="#94A3B8", linewidth=1.5))
            libre = s != _META and s not in _POZOS
            if Q is None or not libre:
                eje.text(c + 0.5, f + 0.5, texto, ha="center", va="center", color=color,
                         fontsize=14, fontweight="bold")
                continue
            if s == _INICIO:
                eje.text(c + 0.5, f + 0.14, "Salida", ha="center", va="center", color=color, fontsize=10,
                         fontweight="bold")
            if np.allclose(Q[s], Q[s][0]):
                eje.plot(c + 0.5, f + 0.45, "o", color="#94A3B8", markersize=7)
            else:
                df, dc = _MOV[int(np.argmax(Q[s]))]
                eje.annotate("", xy=(c + 0.5 + 0.27 * dc, f + 0.45 + 0.22 * df),
                             xytext=(c + 0.5 - 0.27 * dc, f + 0.45 - 0.22 * df),
                             arrowprops=dict(arrowstyle="-|>", color="#8A2E9E", lw=3, mutation_scale=22))
            if visitas is not None:
                eje.text(c + 0.5, f + 0.84, f"{int(visitas[s])}", ha="center", va="center", color="#475569",
                         fontsize=11)
    eje.set_xlim(0, _COLUMNAS)
    eje.set_ylim(_FILAS, 0)
    eje.set_aspect("equal")
    eje.set_xticks([x + 0.5 for x in range(_COLUMNAS)], [str(x) for x in range(_COLUMNAS)])
    eje.set_yticks([y + 0.5 for y in range(_FILAS)], [str(y) for y in range(_FILAS)])
    eje.tick_params(length=0)
    for lado in ("left", "bottom"):
        eje.spines[lado].set_visible(False)
    eje.set_xlabel("Columna")
    eje.set_ylabel("Fila")
    if titulo:
        eje.set_title(titulo, fontsize=13)
    plt.show()


def dibujar_planes(planes, titulo=""):
    """Una fila por política: el color de cada mes dice qué acción toma (nada, llamar u oferta)."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch, Rectangle

    colores = {0: "#E2E8F0", 1: "#8A2E9E", 2: "#F46201"}
    nombres = {0: "Nada", 1: "Llamar", 2: "Oferta de retención"}
    fig, eje = plt.subplots(figsize=(10.5, 0.75 * len(planes) + 1.5))
    for i, (nombre, acciones) in enumerate(planes.items()):
        for mes, a in enumerate(acciones, start=1):
            eje.add_patch(Rectangle((mes - 0.5, i - 0.4), 1, 0.8, facecolor=colores[a], edgecolor="#FFFFFF",
                                    linewidth=1.5))
    eje.set_xlim(0.5, len(next(iter(planes.values()))) + 0.5)
    eje.set_ylim(len(planes) - 0.5, -0.5)
    eje.set_yticks(range(len(planes)), list(planes))
    eje.set_xticks([1, 5, 10, 12, 18, 24])
    eje.set_xlabel("Mes del cliente")
    eje.tick_params(length=0)
    for lado in ("left", "bottom"):
        eje.spines[lado].set_visible(False)
    eje.axvspan(9.5, 12.5, ymin=0, ymax=1, facecolor="none", edgecolor="#1E293B", linewidth=1.2, linestyle="--")
    eje.legend(handles=[Patch(facecolor=colores[a], label=nombres[a]) for a in (0, 1, 2)], loc="upper center",
               bbox_to_anchor=(0.5, -0.32 if len(planes) > 2 else -0.5), ncol=3, frameon=False)
    if titulo:
        eje.set_title(titulo, fontsize=13)
    plt.show()

# Descarga datos/clientes.csv: los mismos 8000 clientes de las sesiones 1 a 4
ruta = descargar_datos("datos")

---
# Parte 1 · Qué cambia

En la sesión 2, cada cliente traía su **respuesta correcta** (si canceló o no) y el modelo aprendía a
imitarla. En la sesión 3 no había respuesta: buscábamos grupos. Hoy, otra forma de aprender: un **agente**
toma decisiones en un **entorno**, y nadie le dice cuál era la correcta. Solo recibe una **recompensa**,
a veces mucho después, que depende de lo que hizo.

- El **estado** es lo que el agente ve antes de decidir (dónde está, en qué mes va el cliente).
- La **acción** es lo que elige; la **política** es su regla para elegir en cada estado.
- La **recompensa** es un número que el entorno le devuelve después de cada acción.
- Un **episodio** es una partida completa, de la salida al final.
- El **retorno** es la suma de recompensas de un episodio. Con un **factor de descuento** γ (gamma), lo
  lejano vale menos: $G = R_1 + \gamma R_2 + \gamma^2 R_3 + \dots$

El entorno puede ser **azaroso**: la misma acción no siempre da lo mismo. Por eso se habla de lo que el
agente **espera** ganar, el promedio de muchas partidas. Y hay un dilema: **explotar** lo que ya sabe o
**explorar** algo nuevo que quizá sea mejor.

---
# Parte 2 · Q-learning en una rejilla

## 1 · El entorno: una rejilla de 4 × 4

El agente sale de la casilla de abajo a la izquierda (fila 3, columna 0) y quiere llegar a la meta, arriba a
la derecha, que da **+10**. Hay dos pozos que dan **−10**. Llegar a la meta o caer en un pozo termina el
episodio. Las demás casillas dan 0: **no hay costo por paso**, y lo que hace preferir el camino corto es γ.
Las acciones son 0 (arriba), 1 (derecha), 2 (abajo) y 3 (izquierda); si choca con el borde, se queda donde
está.

In [ ]:
# [rejilla]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILAS, COLUMNAS = 4, 4
INICIO, META = (3, 0), (0, 3)
POZOS = [(1, 1), (2, 3)]
MOVIMIENTOS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # ↑ → ↓ ←

def paso(s, a):
    """El entorno: desde la casilla s, con la acción a → (s2, r, fin)."""
    f = min(max(s[0] + MOVIMIENTOS[a][0], 0), FILAS - 1)
    c = min(max(s[1] + MOVIMIENTOS[a][1], 0), COLUMNAS - 1)
    s2 = (f, c)
    if s2 == META:
        return s2, 10, True       # llegó: +10 y termina
    if s2 in POZOS:
        return s2, -10, True      # cayó: −10 y termina
    return s2, 0, False           # sigue, sin recompensa

dibujar_rejilla(titulo="La rejilla")

## 2 · Un agente al azar

Antes de aprender nada, ¿qué pasa si el agente elige cada acción al azar? Lo probamos con 10 000 episodios,
con un tope de 50 pasos por episodio.

In [ ]:
# [azar]
rng = np.random.default_rng(2026)
final = {"pozo": 0, "meta": 0, "tope de 50 pasos": 0}
for episodio in range(10_000):
    s, fin = INICIO, False
    for t in range(50):
        s, r, fin = paso(s, int(rng.integers(4)))  # una acción al azar
        if fin:
            break
    if not fin:
        final["tope de 50 pasos"] += 1
    else:
        final["meta" if s == META else "pozo"] += 1
for como, cuantos in final.items():
    print(f"{como}: {cuantos / 10_000:.1%}")

## 3 · La tabla Q y la regla de Q-learning

El agente guarda una **tabla Q**: para cada casilla y cada acción, **lo que espera ganar si hace esto aquí**,
con lo que vale menos lo lejano. Al principio todo vale 0. Después de cada paso, mueve ese valor un poquito
hacia lo que acaba de ver (Sutton y Barto, ecuación 6.8):

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,\big]$$

- $r + \gamma \max Q(s', \cdot)$ es el **objetivo**: la recompensa que recibió más lo mejor que espera ganar
  desde la casilla nueva, descontado. Si el paso terminó el episodio, ya no hay después: ese término vale 0.
- α (alfa) es la **tasa de aprendizaje**, como en la sesión 4: qué fracción del camino hacia el objetivo
  avanza en cada paso.

Dos pasos a mano, con α = 0,1, γ = 0,9 y la tabla en ceros: en un episodio, el agente va a la derecha desde
(0, 2) y llega a la meta; en un episodio posterior, va a la derecha desde (0, 1) y llega a (0, 2).

In [ ]:
# [a_mano]
alfa, gamma = 0.1, 0.9
# 1. Desde (0, 2) va a la meta: r = 10 y termina (no hay después)
q_meta = 0 + alfa * (10 + 0 - 0)
# 2. Desde (0, 1) va a (0, 2): r = 0,
#    y lo mejor que espera ganar en (0, 2) ya es q_meta
q_antes = 0 + alfa * (0 + gamma * q_meta - 0)
print(f"Q[(0, 2), derecha] = {q_meta:.2f}")
print(f"Q[(0, 1), derecha] = {q_antes:.2f}")

**La recompensa viaja hacia atrás.** La casilla (0, 1) nunca tocó la meta, pero ya sabe que ir a la derecha
vale algo, porque desde (0, 2) se llega a la meta. Con más episodios, el valor sigue retrocediendo, casilla a casilla,
hasta la salida.

---
## Ejercicio 1 · La actualización de Q

Completa la función `actualiza`: recibe la tabla `Q`, la casilla `s`, la acción `a`, la recompensa `r`, la
casilla nueva `s2` y si el episodio terminó (`fin`), y **devuelve el valor nuevo de `Q[s][a]`** según la regla
de arriba. No cambies la primera línea (la de `def`). El verificador la prueba con casos que no son de la
rejilla, y si algo no cuadra te dice qué parte de la regla revisar.

In [ ]:
# [ej1]
def actualiza(Q, s, a, r, s2, fin, alfa=0.1, gamma=0.9):
    """Devuelve el valor nuevo de Q[s][a] (no hace falta cambiar Q)."""
    lo_mejor_despues = 0 if fin else Q[s2].max()  # si terminó, no hay más
    objetivo = ...  # la recompensa más lo mejor de después, descontado
    return ...      # el valor actual avanza una fracción alfa hacia él

revisar_actualiza(actualiza)

## 4 · Entrenar: explorar un poco, explotar casi siempre

En cada paso, el agente sigue la **regla ε-codiciosa** (*ε-greedy*): con probabilidad ε elige una acción al
azar (**explora**); el resto de las veces, la de mayor valor en su tabla (**explota**). Si varias empatan,
`np.argmax` elige la primera. Además cuenta cuántas veces pasó por cada casilla (`visitas`).

In [ ]:
# [entrenar]
def entrenar_rejilla(eps=0.1, episodios=500, alfa=0.1, gamma=0.9,
                     semilla=2026):
    rng = np.random.default_rng(semilla)
    Q = np.zeros((FILAS, COLUMNAS, 4))   # al principio, todo vale 0
    visitas = np.zeros((FILAS, COLUMNAS), dtype=int)
    for episodio in range(episodios):
        s = INICIO
        for t in range(50):                  # tope de 50 pasos
            visitas[s] += 1
            if rng.random() < eps:
                a = int(rng.integers(4))     # explora: al azar
            else:
                a = int(np.argmax(Q[s]))     # explota: la mejor
            s2, r, fin = paso(s, a)
            mejor_despues = 0 if fin else Q[s2].max()
            Q[s][a] += alfa * (r + gamma * mejor_despues - Q[s][a])
            s = s2
            if fin:
                break
    return Q, visitas

In [ ]:
# [voraz]
def camino_voraz(Q):
    """Desde la salida, siempre la mejor acción (máx. 50 pasos)."""
    s, camino = INICIO, [INICIO]
    for t in range(50):
        s, r, fin = paso(s, int(np.argmax(Q[s])))
        camino.append(s)
        if fin:
            break
    return camino

def llega_a_la_meta(Q):
    return camino_voraz(Q)[-1] == META

In [ ]:
# [aprende]
Q, visitas = entrenar_rejilla(eps=0.1)
camino = camino_voraz(Q)
print(f"¿Llega? {llega_a_la_meta(Q)}, en {len(camino) - 1} pasos")
print(f"Lo que espera ganar desde la salida: {Q[INICIO].max():.2f}")
print(f"La meta a 6 pasos: 10 · 0.9^5 = {10 * 0.9 ** 5:.2f}")

¿Por qué 0,9⁵? La recompensa llega en el sexto paso, y cada paso que la separa de la salida la multiplica por γ:
la primera acción vale $10\gamma^5$. Si la ruta fuera de 8 pasos, valdría $10\gamma^7$, menos: la ruta corta vale
más. Prueba `entrenar_rejilla(gamma=1.0)`: ¿cuánto vale ahora la salida? Sin γ, todas las rutas que llegan valen
lo mismo.

El dibujo muestra, en cada casilla libre, la acción de mayor valor (la flecha) y cuántas veces pasó el agente
por ella en los 500 episodios. Un punto gris quiere decir que todas sus acciones valen 0: allí no aprendió nada.
Q-learning aprende **fuera de política** (*off-policy*): sus valores apuntan a la mejor política aunque el agente
explore, porque la regla usa el máximo y no la jugada al azar que a veces hace.

In [ ]:
# [flechas]
dibujar_rejilla(Q, visitas, "Lo aprendido con ε = 0,1 (500 episodios)")

Mira las casillas con pocas visitas: con tan pocas pruebas, su flecha no es confiable. El agente aprende bien
lo que recorre a menudo.

---
## Ejercicio 2 · Sin explorar

Entrena dos agentes con la semilla de siempre: uno **sin explorar** (ε = 0) y otro con ε = 0,1. Antes de
ejecutar, **predice** cuál llega a la meta. Después, cuenta cuántas casillas distintas visitó el que no explora
(`casillas_eps0`), mira cuáles son y elige la explicación (la letra, entre comillas, en `razon`):

- **A.** Sin exploración, la tasa de aprendizaje α deja de actuar: la tabla Q se actualiza, pero en pasos tan
  pequeños que no alcanzan a cambiar la decisión.
- **B.** Con γ = 0,9, una meta a seis pasos queda tan descontada que, para el agente, ir hasta ella vale
  menos que quedarse donde está.
- **C.** La tabla empieza en ceros: entre acciones empatadas elige siempre la misma y, como nunca recibe una
  recompensa, nada lo saca de ese camino.
- **D.** 500 episodios no alcanzan para un agente que no explora: con unos miles más, también encontraría la
  ruta hasta la meta.

In [ ]:
# [ej2]
Q_0, visitas_0 = entrenar_rejilla(eps=...)     # no explora nunca
Q_01, visitas_01 = entrenar_rejilla(eps=...)   # explora el 10 %
llega_eps0 = llega_a_la_meta(Q_0)
llega_eps01 = ...                              # ¿y el que explora?
dibujar_rejilla(Q_0, visitas_0, "Sin explorar (ε = 0)")
casillas_eps0 = ...  # cuántas casillas distintas visitó (pista: visitas_0 > 0)
razon = "..."        # la letra de la explicación, entre comillas
revisar_exploracion(llega_eps0=llega_eps0, llega_eps01=llega_eps01,
                    casillas_eps0=casillas_eps0, razon=razon)

---
# Parte 3 · La retención de Yarumo

En la sesión 1 quedó una pregunta para hoy: **¿qué oferta de retención hacer, y cuándo?** Lo planteamos como
un problema de aprendizaje por refuerzo, con un **simulador**: un entorno de mentira con los supuestos a la vista.

- **El episodio** es un cliente nuevo durante 24 meses. **El estado** es el mes del cliente.
- **Las acciones**, cada mes: 0 **nada**, 1 **llamar** o 2 **oferta de retención**.
- **La recompensa**: si el cliente sigue ese mes, Yarumo cobra la factura; la acción cuesta lo que cuesta,
  siga o no. Es facturación, no ganancia, y no se usa γ (equivale a γ = 1): el horizonte es de 24 meses.

## 5 · El riesgo de cancelar sale de los datos

Para cada tramo de antigüedad calculamos la tasa de cancelación a 3 meses (julio a septiembre) de `clientes.csv`. Hasta
el mes 12 tomamos a los clientes **con** el descuento de bienvenida, que vence entre los meses 10 y 12; del 13
al 24, a los que **ya no** lo tienen. La tasa a 3 meses se pasa a un riesgo **mensual** suponiendo que es
parejo en los tres meses: $1 - (1 - p)^{1/3}$.

In [ ]:
# [riesgo]
clientes = pd.read_csv("datos/clientes.csv")
ant = clientes["antiguedad_meses"]
con_bienvenida = (ant <= 12) & (clientes["descuento"] == "Sí")
ya_sin = ant.between(13, 24) & (clientes["descuento"] == "No")
grupo = clientes[con_bienvenida | ya_sin]
tramo_ant = pd.cut(grupo["antiguedad_meses"], [0, 9, 12, 24],
                   labels=["meses 1-9", "meses 10-12", "meses 13-24"])
cancelo = (grupo["cancelo"] == "Sí")
tabla_riesgo = cancelo.groupby(tramo_ant, observed=True).agg(
    clientes="size", a_3_meses="mean")
tabla_riesgo["al_mes"] = 1 - (1 - tabla_riesgo["a_3_meses"]) ** (1 / 3)
RIESGO = tabla_riesgo["al_mes"].to_numpy()
print(tabla_riesgo.round(3))

## 6 · El simulador: lo que sale de los datos y lo que es inventado

La factura es la mediana de `clientes.csv`, en miles de pesos (ficticios). **El efecto y el costo de cada
acción son INVENTADOS**: no hay en los datos ningún experimento que los mida. Supondremos que llamar reduce el
riesgo del mes en una cuarta parte (lo multiplica por 0,75) y cuesta 20 mil pesos, y que la oferta de retención
lo reduce en un 60 % (lo multiplica por 0,4) y cuesta 60 mil.

`valor_politica` calcula **exacto**, sin simular, lo que deja en promedio un cliente con una política, en miles
de pesos. Va del mes 24 hacia el 1: cada mes se paga la acción y, si el cliente no cancela, se cobra la factura
y se suma lo que vale el resto de los meses.

In [ ]:
# [entorno]
FACTURA = round(clientes["factura_mensual"].median() / 1000)  # miles
ACCIONES = ["nada", "llamar", "oferta de retención"]
BAJA_RIESGO = [0, 0.25, 0.60]   # INVENTADO: cuánto baja el riesgo del mes
COSTO = [0, 20, 60]             # INVENTADO: miles de pesos por acción
MESES = 24

def tramo(mes):
    return 0 if mes <= 9 else (1 if mes <= 12 else 2)

def riesgo(mes, accion):
    return RIESGO[tramo(mes)] * (1 - BAJA_RIESGO[accion])

def valor_politica(politica):
    """Lo que deja en promedio un cliente en 24 meses, exacto."""
    valor = 0.0
    for mes in range(MESES, 0, -1):          # de atrás hacia adelante
        a = politica(mes)
        valor = -COSTO[a] + (1 - riesgo(mes, a)) * (FACTURA + valor)
    return valor

print(f"Factura mediana: {FACTURA} mil pesos al mes")

In [ ]:
# [politicas]
politicas = {
    "No hacer nada": lambda mes: 0,
    "Llamar siempre": lambda mes: 1,
    "Oferta siempre": lambda mes: 2,
    "Oferta solo en los meses 10–12": lambda mes: 2 if 10 <= mes <= 12
                                      else 0,
}
for nombre, politica in politicas.items():
    print(f"{nombre:31} {valor_politica(politica):6.1f} mil por cliente")

## 7 · La mejor política posible, porque aquí conocemos el entorno

Como el simulador es nuestro, conocemos el riesgo de cada mes con cada acción, y podemos calcular la mejor
política de atrás hacia adelante: en el mes 24, la mejor acción; en el 23, la mejor sabiendo lo que vale el
24; y así hasta el mes 1. **En la vida real nadie conoce esas probabilidades**: por eso el refuerzo aprende
probando. Este cálculo solo sirve para medir qué tan cerca llega el agente.

In [ ]:
# [mejor]
def mejor_politica():
    valor, mejor = 0.0, {}
    for mes in range(MESES, 0, -1):
        opciones = [-COSTO[a] + (1 - riesgo(mes, a)) * (FACTURA + valor)
                    for a in range(3)]
        mejor[mes] = int(np.argmax(opciones))
        valor = max(opciones)
    return mejor, valor

mejor, valor_mejor = mejor_politica()
print("Mes a mes (0 nada, 1 llamar, 2 oferta):")
print("".join(str(mejor[mes]) for mes in range(1, MESES + 1)))
print(f"La mejor posible: {valor_mejor:.1f} mil por cliente")

## 8 · Q-learning con 20 000 clientes simulados

El agente **no** ve el riesgo: solo prueba acciones con clientes simulados y mira qué pasa. Es la regla de la
rejilla, con tres ajustes: sin γ (equivale a γ = 1); explora mucho al principio (ε baja en línea recta desde 1
y llega a 0,3 en el cliente 7000) y después no baja más, y da pasos α cada vez más cortos en cada mes y acción, porque aquí el
resultado es azaroso y cada cliente simulado trae ruido.

In [ ]:
# [q_yarumo]
def q_learning_yarumo(clientes_sim=20_000, semilla=2026):
    rng = np.random.default_rng(semilla)
    Q = np.zeros((MESES + 2, 3))  # fila = mes, columna = acción
    N = np.zeros((MESES + 2, 3))  # veces que probó cada una
    for k in range(clientes_sim):
        eps = max(0.3, 1 - k / 10_000)        # explora menos con el tiempo
        for mes in range(1, MESES + 1):
            if rng.random() < eps:
                a = int(rng.integers(3))
            else:
                a = int(np.argmax(Q[mes]))
            cancela = rng.random() < riesgo(mes, a)  # el entorno decide
            r = -COSTO[a] + (0 if cancela else FACTURA)
            fin = cancela or mes == MESES
            N[mes, a] += 1
            alfa = max(0.001, 1 / N[mes, a] ** 0.8)  # pasos más cortos
            despues = 0 if fin else Q[mes + 1].max()
            Q[mes, a] += alfa * (r + despues - Q[mes, a])
            if fin:
                break
    return Q

In [ ]:
# [aprendida]
Q_y = q_learning_yarumo()
aprendida = {mes: int(np.argmax(Q_y[mes])) for mes in range(1, MESES + 1)}
print("".join(str(aprendida[mes]) for mes in range(1, MESES + 1)))
valor_aprendida = valor_politica(lambda mes: aprendida[mes])
print(f"Lo aprendido vale {valor_aprendida:.1f} mil por cliente")

In [ ]:
# [planes]
regla = [2 if 10 <= mes <= 12 else 0 for mes in range(1, MESES + 1)]
dibujar_planes({
    "Oferta en 10–12": regla,
    "Q-learning": [aprendida[mes] for mes in range(1, MESES + 1)],
    "La mejor posible": [mejor[mes] for mes in range(1, MESES + 1)],
})

Cuatro cosas, con honestidad:

1. **El agente redescubre lo que el simulador supone.** El valor está en el método: en la vida real, el
   simulador se estima con datos (por ejemplo, de campañas pasadas) y se valida.
2. **Solo lo de los meses 10 a 12 resiste el ruido**, y solo si el efecto inventado de la oferta es cierto.
   En los demás meses, cada mes las acciones valen casi lo mismo y el agente no distingue entre ellas, aunque,
   sumadas, esas pequeñas diferencias son buena parte de lo que separa a la regla de la mejor posible.
3. **«La mejor posible» se calculó porque conocemos el entorno.** El refuerzo sirve cuando no se conoce.
4. **Hicieron falta 20 000 clientes simulados.** No se aprende experimentando así con clientes reales: las
   acciones de exploración rara vez están libres de consecuencias (Dulac-Arnold et al., 2021). Y si una política
   actúa sola, hay que decidir quién la aprueba y qué se registra de cada decisión.

---
## Ejercicio 3 · Tu política de retención

Escribe tu propia política: una función que recibe el mes (de 1 a 24) y devuelve la acción (0 nada, 1 llamar,
2 oferta de retención). Calcula su valor con `valor_politica` (no lo escribas a mano) y compárala con la regla
de los meses 10 a 12 y con la mejor posible. La regla de los meses 10 a 12 tiene tres tramos: puedes agregar o
quitar los que quieras.

In [ ]:
# [ej3]
def mi_politica(mes):
    """Del mes 1 al 24: 0 nada, 1 llamar, 2 oferta de retención."""
    if mes <= ...:
        return ...
    if mes <= ...:       # puedes agregar o quitar tramos
        return ...
    return ...

valor = ...    # calcúlalo con valor_politica
revisar_politica(mi_politica, valor=valor)

---
# Parte 4 · *Policy gradients*: ajustar las probabilidades

Q-learning estima **cuánto vale** cada acción y elige la mejor. Los métodos de **gradiente de política**
(*policy gradients*) no estiman valores: la política es directamente una probabilidad para cada acción, como
la salida **softmax** de la red de dígitos de la sesión 4. Después de cada cliente simulado, **sube la
probabilidad de lo que salió mejor que un valor de referencia y baja la de lo que salió peor**. Es la idea de
REINFORCE (Williams, 1992; Sutton y Barto, ecuaciones 13.8 y 13.11).

Aquí la política tiene tres preferencias por tramo (meses 1–9, 10–12 y 13–24), una por acción. El valor de
referencia es el promedio de lo que se ha ganado desde cada mes; no depende de la acción elegida. Corre la
celda y **mira**: no hay ejercicio.

In [ ]:
# [reinforce]
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def reinforce(semilla, clientes_sim=12_000, alfa=0.005):
    rng = np.random.default_rng(semilla)
    pref = np.zeros((3, 3))           # preferencias: tramo × acción
    referencia = np.zeros(MESES + 1)  # lo que se suele ganar desde cada mes
    historia = []
    for k in range(clientes_sim):
        tramos, acciones, recompensas = [], [], []
        for mes in range(1, MESES + 1):            # un cliente simulado
            pi = softmax(pref[tramo(mes)])
            a = int(rng.choice(3, p=pi))   # elige con probabilidades
            tramos.append(tramo(mes))
            acciones.append(a)
            if rng.random() < riesgo(mes, a):
                recompensas.append(-COSTO[a])
                break
            recompensas.append(FACTURA - COSTO[a])
        # lo que ganó desde cada mes hasta el final, en facturas
        G = np.cumsum(recompensas[::-1])[::-1] / FACTURA
        for t, (k_tramo, a) in enumerate(zip(tramos, acciones)):
            ventaja = G[t] - referencia[t + 1]  # ¿mejor que lo usual?
            referencia[t + 1] += 0.05 * (G[t] - referencia[t + 1])
            pi = softmax(pref[k_tramo])
            sube = -pi
            sube[a] += 1            # la elegida sube; las otras bajan
            pref[k_tramo] += alfa * ventaja * sube
        historia.append([softmax(p) for p in pref])
    return pref, np.array(historia)

In [ ]:
# [reinforce_semillas]
semillas = [2026, 1, 2, 3, 4]
fig, ejes = plt.subplots(1, 2, figsize=(10.5, 3.6), sharey=True)
for semilla in semillas:
    pref, historia = reinforce(semilla)
    x = np.arange(0, len(historia), 40)          # un punto cada 40 clientes
    ejes[0].plot(x, historia[x, 1, 2], label=f"semilla {semilla}")
    ejes[1].plot(x, historia[x, 0, 1])
    planes = [ACCIONES[int(np.argmax(p))] for p in pref]
    print(f"Semilla {semilla}: 1-9 {planes[0]}, 10-12 {planes[1]}, "
          f"13-24 {planes[2]}")
ejes[0].set_title("Meses 10–12: P(oferta)")
ejes[1].set_title("Meses 1–9: P(llamar)")
for eje in ejes:
    eje.set_xlabel("Clientes simulados")
ejes[0].legend(fontsize=10)
plt.show()

En los meses 10 a 12 (y, en estas cinco, también en los 13 a 24) todas las semillas aprenden lo mismo. En los
meses 1 a 9, la diferencia entre las acciones es pequeña frente al ruido de cada cliente simulado, y lo que se
aprende cambia de una semilla a otra: estos métodos pueden tener **varianza alta**. El valor de referencia la
reduce, pero no la elimina.

Un método de esta familia, PPO (Schulman et al., 2017), se usó para ajustar InstructGPT con las preferencias
de personas (Ouyang et al., 2022): allí la recompensa la da un modelo entrenado con comparaciones hechas por personas.

---
### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: en la regla de Q-learning, ¿qué término hace que la recompensa de la meta llegue, con los
   episodios, hasta la salida? …
2. Ejercicio 2: el agente que no explora visitó … casillas distintas. ¿Por qué no llega? ¿Se parece a una
   empresa que solo repite lo que ya le funcionó? …
3. Ejercicio 3: mi política fue … y vale … mil pesos por cliente; la regla de los meses 10–12 vale … y la mejor
   posible, … ¿Por qué la tuya gana o pierde? ¿Qué supuesto del simulador, si cambiara, cambiaría tu
   respuesta? …
4. Gradiente de política: ¿en qué tramos coinciden las cinco semillas y en cuál no? ¿Por qué crees que pasa? …
5. Si Yarumo dejara que una política así decidiera sola, ¿quién debería aprobarla y qué registrarías de cada
   decisión? …

---
## La respuesta a la gerencia

**¿Qué oferta de retención hacer, y cuándo?** Bajo los supuestos de este simulador, la oferta vale la pena
cuando vence el descuento de bienvenida (meses 10 a 12), que es cuando más cancelan; en los demás meses, la
diferencia entre llamar y no hacer nada es pequeña. El agente lo aprendió con 20 000 clientes **simulados**: el
siguiente paso no es probarlo con clientes reales, sino estimar el efecto de cada acción con datos de campañas
y validar el simulador. Si una política decide sola, alguien la aprueba y cada decisión queda registrada.